# LAB 4G – ĐÁNH GIÁ BA HƯỚNG TỔNG HỢP TIẾNG NÓI

Dùng output thực của A–F và tài liệu Lab 4 mục 4.7, 5.7, 6, 7, 8. **Không suy diễn điểm MOS**, không gọi trajectory của HMM là waveform. Có thể chạy notebook này trước khi nghe để tạo bài nghe mù; chạy lại sau khi điền điểm để có bảng và hình so sánh.

## 1. Kiểm tra đầu vào và môi trường

Bước F cần hoàn thành trước để có `evaluation/f_neural_latency_comparison.csv`, 3 WAV neural và 3 WAV ghép nối từ `_01` + cross-fade. Nếu không có model offline thật, dừng phần đánh giá neural.

In [1]:
from pathlib import Path
import re, random, shutil, json, statistics, time, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import display, Audio
ROOT=Path.cwd().resolve(); AUDIO=ROOT/'audio_output'; EVAL=ROOT/'evaluation'; FIG=ROOT/'figures'
for folder in (AUDIO,EVAL,FIG): folder.mkdir(exist_ok=True)
metric_path=EVAL/'f_neural_latency_comparison.csv'
if not metric_path.exists():
    raise FileNotFoundError('Chạy Lab04_Buoc6_F_OFFLINE.ipynb trước khi chạy phần G')
metrics=pd.read_csv(metric_path)
assert len(metrics)==3 and (metrics['latency_s']>0).all()
print('Neural inference metrics từ F:')
display(metrics[['id','spoken_text','neural_duration_s','latency_s','RTF','voice']])

Neural inference metrics từ F:


,id,spoken_text,neural_duration_s,latency_s,RTF,voice
0,test01,Mời khách hàng số hai ba đến quầy năm.,2.08,1.9163,0.9213,SDK default
1,test02,Mời khách hàng số không bảy đến quầy ba.,2.00,1.5601,0.7801,SDK default
2,test03,Mời khách hàng số một tám đến quầy hai.,2.00,1.5408,0.7704,SDK default


## 2. Đánh giá khách quan và giới hạn của HMM

`latency_s` của neural đo sau model initialization (không tính load weights). `RTF` = latency / duration. Đường HMM ở phần E chỉ sinh tham số; không chấm MOS/latency tổng hợp waveform nếu chưa có vocoder. Unit selection phần C có thể không thay đổi path khi baseline đã là tối ưu.

In [2]:
print('Mean neural inference latency:', round(float(metrics['latency_s'].mean()),3),'s')
print('Mean RTF:', round(float(metrics['RTF'].mean()),3))
hmm_path=ROOT/'e_hmm_parameter_experiments.csv'
if not hmm_path.exists(): hmm_path=EVAL/'e_hmm_parameter_experiments.csv'
if hmm_path.exists():
    hmm=pd.read_csv(hmm_path)
    print('HMM từ phần E (tham số, KHÔNG phải waveform):')
    display(hmm)
else:
    print('Chưa có e_hmm_parameter_experiments.csv trong dự án, có thể bổ sung lúc viết báo cáo.')
cost_path=ROOT/'c_total_cost_comparison.csv'
if cost_path.exists():
    print('Unit selection từ phần C:')
    display(pd.read_csv(cost_path))

Mean neural inference latency: 1.672 s
Mean RTF: 0.824
Chưa có e_hmm_parameter_experiments.csv trong dự án, có thể bổ sung lúc viết báo cáo.


## 3. Tạo bài nghe mù cho ít nhất 5 người nghe

Để tránh bias, copy 6 WAV sang tên `S01..S06` và đặt danh sách chấm **không có tên phương pháp**. Giữ `g_blind_key.csv` riêng không cung cấp cho người nghe. Mỗi người nghe cả 6 file và tự điền 3 tiêu chí từ 1–5; ghi lại nội dung họ nghe được trong `heard_text`. Có thể làm nhiều hơn 5 người.

In [3]:
blind_dir=EVAL/'blind_audio'; blind_dir.mkdir(exist_ok=True)
source=[]
for _,r in metrics.iterrows():
    for method,column in [('concat','concat_audio_path'),('neural','audio_path_16k')]:
        p=Path(r[column]);
        if not p.exists(): raise FileNotFoundError(f'Missing audio: {p}')
        source.append(dict(test_id=r['id'],method=method,source_file=str(p),reference_text=r['spoken_text']))
random.Random(457).shuffle(source)
key=[]
for index,row in enumerate(source,1):
    clip=f'S{index:02d}'; destination=blind_dir/f'{clip}.wav'
    shutil.copyfile(row['source_file'], destination)
    key.append(dict(sample_id=clip,**row,audio_for_listening=str(destination)))
key=pd.DataFrame(key)
key.to_csv(EVAL/'g_blind_key.csv',index=False,encoding='utf-8-sig')
ratings_path=EVAL/'g_ratings.csv'
if not ratings_path.exists():
    blank=[dict(listener_id=f'L{listener:02d}',sample_id=row['sample_id'],
      intelligibility_1_5='',naturalness_1_5='',continuity_1_5='',heard_text='',notes='')
      for listener in range(1,6) for _,row in key.iterrows()]
    pd.DataFrame(blank).to_csv(ratings_path,index=False,encoding='utf-8-sig')
print('Audio nghe mù:',blind_dir)
print('Bảng chấm:',ratings_path)
print('Đừng mở g_blind_key.csv cho người nghe trước khi họ chấm.')
display(key[['sample_id','test_id']].sort_values('sample_id'))

Audio nghe mù: C:\xulyamthanh\lab04_2351260688_NguyenNgocTien\evaluation\blind_audio
Bảng chấm: C:\xulyamthanh\lab04_2351260688_NguyenNgocTien\evaluation\g_ratings.csv
Đừng mở g_blind_key.csv cho người nghe trước khi họ chấm.


,sample_id,test_id
0,S01,test01
1,S02,test02
2,S03,test03
3,S04,test03
4,S05,test02
5,S06,test01


## 4. Nghe thử và ghi nhận nội dung người nghe

Mỗi người nên nghe bằng cùng tai nghe/loa, tại mức âm lượng dễ chịu; âm lượng hai phương pháp nên được cân bằng trước khi đánh giá. Nhập điểm trong `evaluation/g_ratings.csv` rồi chạy lại các cell từ đây. Nếu chưa có điểm, notebook sẽ không giả lập kết quả. `heard_text` để tính khả năng hiểu đúng từ bằng WER.

In [4]:
for file in sorted(blind_dir.glob('S*.wav')):
    print(file.name)
    display(Audio(filename=str(file)))

S01.wav


S02.wav


S03.wav


S04.wav


S05.wav


S06.wav


## 5. Tính MOS và intelligibility bằng dữ liệu đã chấm

- Điểm naturalness, continuity và intelligibility là trung bình đánh giá của người nghe (1–5).
- Word intelligibility (%) = 100×max(0,1−WER) khi có transcription `heard_text`.
- Chỉ tổng hợp **các dòng có đủ ba điểm**. Nếu số người đánh giá quá ít, ghi rõ hạn chế.

In [5]:
ratings=pd.read_csv(ratings_path, dtype={'listener_id':'string','sample_id':'string'})
scales=['intelligibility_1_5','naturalness_1_5','continuity_1_5']
for col in scales: ratings[col]=pd.to_numeric(ratings[col],errors='coerce')
valid=ratings[ratings[scales].notna().all(axis=1)].copy()
valid=valid[valid[scales].ge(1).all(axis=1) & valid[scales].le(5).all(axis=1)]
valid=valid.merge(key[['sample_id','test_id','method','reference_text']],on='sample_id',how='inner')

def tokens(s): return re.findall(r'\w+',str(s).casefold(),flags=re.UNICODE)
def edits(ref,hyp):
    a,b=tokens(ref),tokens(hyp)
    old=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        new=[i]
        for j,y in enumerate(b,1):
            new.append(min(new[-1]+1,old[j]+1,old[j-1]+(x!=y)))
        old=new
    return old[-1],len(a)
if len(valid):
    valid['word_correct_pct']=valid.apply(lambda r: max(0,100*(1-edits(r['reference_text'],r['heard_text'])[0]/max(1,edits(r['reference_text'],r['heard_text'])[1]))) if pd.notna(r['heard_text']) and str(r['heard_text']).strip() else np.nan,axis=1)
    summary=valid.groupby('method',as_index=False).agg(
        n_ratings=('listener_id','count'),
        n_listeners=('listener_id','nunique'),
        intelligibility_rating=('intelligibility_1_5','mean'),
        naturalness_MOS=('naturalness_1_5','mean'),
        continuity_MOS=('continuity_1_5','mean'),
        word_correct_pct=('word_correct_pct','mean'))
    summary.to_csv(EVAL/'g_listening_summary.csv',index=False,encoding='utf-8-sig')
    display(summary)
    fig,ax=plt.subplots(figsize=(8,4))
    summary.plot.bar(x='method',y=['naturalness_MOS','continuity_MOS','intelligibility_rating'],ax=ax,rot=0)
    ax.set(ylabel='Mean listener score 1–5',title='MOS từ dữ liệu người nghe thực')
    ax.set_ylim(0,5); fig.tight_layout(); fig.savefig(FIG/'g_listening_scores.png',dpi=150)
    plt.show()
else:
    summary=pd.DataFrame()
    print('CHƯA có bài nghe chấm điểm: MOS và % đúng từ để trống, không tự điền.')

CHƯA có bài nghe chấm điểm: MOS và % đúng từ để trống, không tự điền.


## 6. Xuất bảng so sánh cuối cùng của Lab 4

Bảng gồm 3 hướng tổng hợp. Neural có latency đo được; concatenation có audio và chi phí candidate (không dùng latency neural để gán cho concat); HMM có trajectory nhưng chưa có waveform. Naturalness/intelligibility chỉ gán khi có đánh giá người nghe.

In [6]:
rows=[]
for method,label in [('concat','Concatenative (unit selection + cross-fade)'),('hmm','HMM/parametric (trajectory only)'),('neural','Neural VieNeu pretrained offline')]:
    r={'method':label,'waveform_available':method!='hmm',
       'latency_s':round(float(metrics['latency_s'].mean()),4) if method=='neural' else np.nan,
       'RTF':round(float(metrics['RTF'].mean()),4) if method=='neural' else np.nan,
       'naturalness_MOS':np.nan,'continuity_MOS':np.nan,'intelligibility_rating':np.nan,
       'word_correct_pct':np.nan,
       'data_memory':'16 word units × 2 candidates' if method=='concat' else ('HMM state/parameter files' if method=='hmm' else 'Local pretrained weights (size to measure)'),
       'limitation':'No waveform/vocoder: listening comparison N/A' if method=='hmm' else ''}
    if len(summary) and method in set(summary['method']):
        s=summary[summary['method']==method].iloc[0]
        for field in ['naturalness_MOS','continuity_MOS','intelligibility_rating','word_correct_pct']:
            r[field]=s[field]
    rows.append(r)
final=pd.DataFrame(rows)
final.to_csv(EVAL/'g_three_methods_comparison.csv',index=False,encoding='utf-8-sig')
display(final)

,method,waveform_available,latency_s,RTF,naturalness_MOS,continuity_MOS,intelligibility_rating,word_correct_pct,data_memory,limitation
0,Concatenative (unit selection + cross-fade),True,NaN,NaN,NaN,NaN,NaN,NaN,16 word units × 2 candidates,
1,HMM/parametric (trajectory only),False,NaN,NaN,NaN,NaN,NaN,NaN,HMM state/parameter files,No waveform/vocoder: listening comparison N/A
2,Neural VieNeu pretrained offline,True,1.6724,0.8239,NaN,NaN,NaN,NaN,Local pretrained weights (size to measure),


## 7. Ghi báo cáo và kết luận có căn cứ

- Concatenation: word-unit 16 từ/2 candidate, DP từ phần C; cross-fade 0–30ms.
- HMM: state-parameter generation và delta/MLPG, không báo MOS vì chưa có waveform.
- Neural: pretrained VieNeu-TTS chạy **offline**, đo inference latency sau khi load model và tạo WAV/mel cho 3 câu.
- Khẳng định về chất lượng phải dựa trên điểm nghe của người nghe thật, không suy diễn từ phổ.
- Đưa bảng `g_three_methods_comparison.csv` và minh chứng Terminal chạy offline vào báo cáo 5–8 trang.